# HARA — Verifier Training on Google Colab
End-to-end pipeline for the **full HotpotQA distractor train split (90,447 questions)**:
BUILD → LABEL → TRAIN, with everything session-proof.

**Before you start**
1. Runtime → Change runtime type → **GPU** (T4 on free tier; L4/A100 on Pro are much faster — see timings at the bottom).
2. Upload the **optimized HARA `.py` files** to Google Drive at `MyDrive/hara/code/` (one-time). The notebook needs the new `--resume`, `--save-every-steps`, and `--num-shards` flags, which the optimized set adds.
3. Have ~5 GB free on Drive (datasets JSONL + checkpoints live there so nothing is lost when a session ends).

**Session-limit playbook:** when Colab disconnects, just reconnect and run every cell top-to-bottom again. BUILD, LABEL, the premise cache, and TRAIN are all resumable — they continue where they stopped.

In [ ]:
# 1) What GPU did we get? (T4 = no bf16 -> training falls back to fp32; L4/A100 support bf16)
!nvidia-smi
import torch
print("bf16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else "no CUDA")

In [ ]:
# 2) Mount Google Drive — all persistent state lives here
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/hara'
for sub in ('', '/code', '/models'):
    os.makedirs(DRIVE + sub, exist_ok=True)
print('Persistent dir:', DRIVE)

In [ ]:
# 3) Copy the pipeline code from Drive into the fast local runtime
#    (Alternative: git clone your repo — but only after the optimized files
#     with the new flags are pushed to the Manseez branch.)
import glob, shutil
py_files = glob.glob(f'{DRIVE}/code/*.py')
assert py_files, f'No .py files found in {DRIVE}/code — upload the optimized HARA files there first.'
os.makedirs('/content/hara', exist_ok=True)
for p in py_files:
    shutil.copy(p, '/content/hara/')
%cd /content/hara
print(f'{len(py_files)} files ready')

In [ ]:
# 4) Dependencies (torch ships with Colab; sentencepiece/protobuf are required for the DeBERTa tokenizer)
!pip -q install "transformers>=4.56" peft sentence-transformers faiss-cpu datasets ollama rank-bm25 sentencepiece protobuf

In [ ]:
# 5) Install Ollama + pull the generator (needed for BUILD and LABEL; skip this cell if you
#    uploaded a finished labeled dataset and only want to TRAIN)
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull llama3.2
!ollama list

## BUILD — generate candidate answers for all 90,447 train questions
Writes straight to Drive and **resumes automatically** (already-processed question IDs are skipped), so re-running this cell after a disconnect continues where it stopped.

**Running your laptop in parallel?** Give each machine a disjoint shard — the partition is deterministic by question-id hash, so there's zero coordination:
- Laptop: `--num-shards 2 --shard-index 0 --output verifier_dataset_shard0.jsonl`
- Colab (edit the cell below): `--num-shards 2 --shard-index 1`

Watch the tqdm rate for ~10 minutes: `seconds/question × 90,447` is your real total.

In [ ]:
!python Stage_2_Dataset.py build \
    --num-samples 90447 \
    --output {DRIVE}/verifier_dataset_colab.jsonl
# Parallel-with-laptop version:  add  --num-shards 2 --shard-index 1

## LABEL — assign SUPPORTED / PARTIAL / UNSUPPORTED
If you built in shards on two machines, first concatenate the shard files into one (order doesn't matter; the label step keys records individually and is itself resumable).

In [ ]:
# If sharded: merge laptop's file (upload it to Drive first) with Colab's before labeling
# !cat {DRIVE}/verifier_dataset_shard0.jsonl {DRIVE}/verifier_dataset_colab.jsonl > {DRIVE}/verifier_dataset_full.jsonl

!python Stage_2_Dataset.py label \
    --input  {DRIVE}/verifier_dataset_colab.jsonl \
    --output {DRIVE}/verifier_dataset_labeled.jsonl

## TRAIN — LoRA fine-tune DeBERTa-v3-large
Session-proof: `--save-every-steps 500` checkpoints adapter + optimizer + scheduler + exact batch position to Drive every 500 optimizer steps, and `--resume` picks it up — a rerun fast-forwards to the precise batch it stopped at (fast-forwarding costs minutes, not hours). The first run also builds the premise cache (long cross-encoder pass, one-time, cached on Drive).

**Precision:** on T4 the script auto-falls back from bf16 to fp32 (safe, slower). Optionally try `--precision fp16` — it keeps fp32 master weights so LoRA at lr 1e-5 usually tolerates it — but watch the loss: if it goes NaN, remove the flag. On L4/A100, bf16 just works.

In [ ]:
!python Stage_2_Verifier_Train.py \
    --data-path    {DRIVE}/verifier_dataset_labeled.jsonl \
    --premise-cache {DRIVE}/verifier_train_premises_cache.jsonl \
    --output-dir   {DRIVE}/models/hara_deberta_v3_large_verifier \
    --epochs 4 --patience 2 --seed 42 --class-weights auto \
    --batch-size 16 --grad-accum 2 \
    --save-every-steps 500 --resume

## Afterwards — get the checkpoint onto your laptop
The finished model (plus `training_metadata.json` with the per-class table and confusion matrix for the report) is already on Drive at `MyDrive/hara/models/hara_deberta_v3_large_verifier/`. Sync/download that folder into the project's `models/hara_deberta_v3_large_verifier/` on your laptop — every stage picks it up automatically. You can delete `resume_state.pt` from the folder once training is done.

## Rough timings (full 90k questions ≈ 0.7–1.1 M training pairs)
| Phase | T4 (free) | L4 / A100 (Pro) | Notes |
|---|---|---|---|
| BUILD | ~4–8 s/question → 100–200 h | ~2.5–4 s/q → 60–100 h | Ollama-bound; halve wall-clock by sharding with the laptop |
| LABEL | ~1–3 days machine time | somewhat less | Deterministic majority fast; LLM fallback dominates |
| Premise cache | several hours – 1 day | a few hours | One-time, cached on Drive |
| TRAIN / epoch | ~12–20 h fp32 (fp16 ≈ 6–9 h if stable) | L4 ≈ 3–5 h · A100 ≈ 1–2 h | Patience usually stops at 2–3 epochs |

Free-tier sessions cap around 12 h with weekly quotas, so plan on many sessions — the resume machinery exists precisely so that's an inconvenience, not a risk. A cheaper first move: train checkpoints at ~10k and ~30k questions from the same growing dataset and plot macro F1 vs. size — if the curve is flat by 30k, you have your report figure *and* your answer about whether the full 90k is worth the remaining compute.